# Higher-order structure over time — does the meta crystallize?

**Hypothesis.** Early in a regulation, teambuilding is exploratory and close to "pick good
pieces roughly independently", which produces weak higher-order correlations and a
*pairwise-sufficient* composition distribution. As archetypes crystallize, genuine three-body
structure (slot competition, archetype cores) should *emerge*.

Reg **M-A** is the ideal test bed: a brand-new format in a brand-new game, with no prior similar
meta to inherit structure from. We fit a **Boltzmann (moment-matched)** model on a **sliding
window** of M-A and track higher-order structure as the window advances in time. `MODEL` selects
**species+item** (the default — item-resolved, where most of the structure lives) or the smaller
**species** model we calibrated on.

**Methodology — the traps we control for:**
1. **Statistical power.** The `|z|>3` fraction conflates *how much* structure exists with *how
   resolvable* it is, and resolvability grows with sample size. So we use **equal-team-count
   windows** (not equal-time), holding N — and therefore the data sampling error — constant.
2. **Confounding by overall structuring.** A pairwise model *generates* 3-point correlations by
   propagation, so the data's raw 3-point grows just from the meta getting more structured. The
   de-confounded signal is `irreducible` (data 3-point minus the moment-matched pairwise model's
   3-point) — what *no* pairwise model can produce. That, not raw `struct`, is the emergence test.
3. **Fit quality.** Sharper late-meta windows are harder to moment-match, and any 2-point miss
   leaks into the 3-point residual as fake "irreducible". A per-window 2-point fit-residual
   **control** must stay flat for the irreducible trend to mean anything.
4. **Null.** A shuffled-time control (windows over a random team order) must come out flat.

Per-window pair fits use the **full** coupling set (no support-gating; L2 reg stabilizes thin
pairs), so all 2-point moments are matched and the 3-point test stays clean.

In [ ]:
from __future__ import annotations

import datetime
from collections import Counter
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from k2dex import tournament_ingest
from k2dex.constants import TEAM_SIZE, SPECIES_LR_LAMBDA, SPECIES_ITEM_LR_LAMBDA
from k2dex.loaders import format_pair
from k2dex.models import fit_pl_ising, fit_boltzmann_ising, empirical_moments
from k2dex.sampling import parallel_tempered_mcmc

REG = "M-A"
# Where most of the structure lives. Set to "species" for the smaller/faster
# model (what we calibrated on). Per-window pair fits use the full coupling set
# (no support-gating; the L2 reg stabilizes thin pairs), so the 2-point moments
# are all matched and the 3-point test stays clean.
MODEL = "species_item"
LAM = SPECIES_ITEM_LR_LAMBDA if MODEL == "species_item" else SPECIES_LR_LAMBDA

## Helpers

`window_metrics` runs the full pipeline on one window's teams: build the species design matrix
(uniform weights — the window *is* the time slice, so no recency reweighting), PL warm-start →
Boltzmann fit, then connected 3-point correlations of data vs the fitted model over the window's
own top-`K_TOP` features. It returns both the model-free structure magnitude and the
pairwise-insufficiency metrics.

In [ ]:
def triple_product_mean(A, W):
    K = A.shape[1]
    out = np.empty((K, K, K))
    Aw = A * W[:, None]
    for i in range(K):
        out[i] = (A * Aw[:, [i]]).T @ A
    return out


def connected_moments(X, w, idx):
    # weighted connected 3-point tensor T_ijk and its per-triplet standard error
    W = w / w.sum()
    Xc = X[:, idx].astype(np.float64)
    Xc = Xc - W @ Xc
    T = triple_product_mean(Xc, W)
    G2 = triple_product_mean(Xc**2, W)
    se = np.sqrt(np.clip((W @ W) * (G2 - T**2), 0.0, None))
    return T, se


def sample_model(J, h, *, species_of=None, item_of=None,
                 n_runs=8, n_steps=30_000, burn_in=6_000, thin=20, seed0=0):
    ladder = np.geomspace(1.0, 3.0, 8)   # cold chain at T=1 samples P ∝ exp(-H)
    rng = np.random.default_rng(seed0)
    pooled = []
    for _ in range(n_runs):
        res = parallel_tempered_mcmc(J, h, TEAM_SIZE, [], [], 1.0, ladder,
                                     n_steps, burn_in, 10, int(rng.integers(2**31)),
                                     species_of=species_of, item_of=item_of)
        assert res is not None
        pooled.append(res[0][::thin])
    return np.concatenate(pooled).astype(np.float64)


def build_window(obs_window, min_count=5):
    """Returns (vocab, X, species_of, item_of). Species model has no uniqueness
    lookups (None, None); species+item carries the no-dup-species/item lookups."""
    if MODEL == "species":
        teams = tournament_ingest.species_only_teams([o.members for o in obs_window])
        raw = Counter(n for t in teams for n in t)
        keys = sorted(n for n, c in raw.items() if c >= min_count)
        idx = {k: i for i, k in enumerate(keys)}
        X = np.zeros((len(teams), len(keys)), dtype=np.int8)
        for ti, team in enumerate(teams):
            for name in team:
                j = idx.get(name)
                if j is not None:
                    X[ti, j] = 1
        return keys, X, None, None

    teams = [o.members for o in obs_window]            # tuples of (species, item)
    raw = Counter(p for t in teams for p in t)
    pairs = sorted((p for p, c in raw.items() if c >= min_count),
                   key=lambda p: format_pair(p[0], p[1]))
    idx = {p: i for i, p in enumerate(pairs)}
    X = np.zeros((len(teams), len(pairs)), dtype=np.int8)
    for ti, team in enumerate(teams):
        for p in team:
            j = idx.get(p)
            if j is not None:
                X[ti, j] = 1
    species_of = [s for s, _ in pairs]
    item_of = [it for _, it in pairs]
    vocab = [format_pair(s, it) for s, it in pairs]
    return vocab, X, species_of, item_of


FIT_KW = dict(reg="l2", reg_lambda=1e-3, n_iters=800, lr=0.01, lr_final=0.0001,
              n_chains=500, n_sweeps=300, n_burn=200, avg_last=50)
SAMPLE_KW = dict(n_runs=25, n_steps=50_000, burn_in=6_000, thin=20)


def window_metrics(obs_window, *, K_TOP=40, seed=0):
    vocab, X, species_of, item_of = build_window(obs_window)
    n = X.shape[0]
    m_data, _ = empirical_moments(X)                       # uniform weights
    J_pl, h_pl = fit_pl_ising(X, C=1.0 / LAM)
    J, h, _ = fit_boltzmann_ising(X, team_size=TEAM_SIZE, init_J=J_pl, init_h=h_pl,
                                  species_of=species_of, item_of=item_of,
                                  seed=seed, progress=False, **FIT_KW)

    idx = np.argsort(m_data)[::-1][:K_TOP]
    trips = np.array(list(combinations(range(K_TOP), 3)))
    ti, tj, tk = trips[:, 0], trips[:, 1], trips[:, 2]
    T_d, se_d = connected_moments(X, np.ones(n), idx)
    S = sample_model(J, h, species_of=species_of, item_of=item_of,
                     seed0=seed + 1, **SAMPLE_KW)
    T_m, se_m = connected_moments(S, np.ones(S.shape[0]), idx)
    td, sd = T_d[ti, tj, tk], se_d[ti, tj, tk]
    tm, sm = T_m[ti, tj, tk], se_m[ti, tj, tk]

    z = (td - tm) / np.sqrt(sd**2 + sm**2 + 1e-24)
    rms = lambda a: float(np.sqrt(np.mean(a**2)))
    noise_d, noise_m = float(np.mean(sd**2)), float(np.mean(sm**2))
    # Noise-debias magnitudes: E[rms(T_hat)^2] ≈ rms(T_true)^2 + mean(se^2).
    struct = float(np.sqrt(max(rms(td)**2 - noise_d, 0.0)))
    irreducible = float(np.sqrt(max(rms(td - tm)**2 - noise_d - noise_m, 0.0)))

    # 2-point fit-quality CONTROL + 2nd-order context, on the same top-K features.
    Xk = X[:, idx].astype(np.float64)
    Sk = S[:, idx]
    Cd2 = (Xk.T @ Xk) / n                          # data raw 2-point
    Cm2 = (Sk.T @ Sk) / S.shape[0]                 # model raw 2-point
    offdiag = ~np.eye(K_TOP, dtype=bool)
    fit_resid2 = float(np.mean(np.abs(Cd2 - Cm2)[offdiag]))   # control: should stay flat
    mk = m_data[idx]
    conn2 = (Cd2 - np.outer(mk, mk))[np.triu_indices(K_TOP, 1)]
    pair2 = float(np.sqrt(np.mean(conn2**2)))                 # 2nd-order magnitude (context)

    return dict(n=n, V=len(vocab),
                rms_data=rms(td), struct=struct, irreducible=irreducible,
                irreducible_frac=irreducible / (struct + 1e-12),
                frac_z3=float(np.mean(np.abs(z) > 3)),
                r=float(np.corrcoef(td, tm)[0, 1]),
                fit_resid2=fit_resid2, pair2=pair2)

## Sliding windows (constant N)

Sort every M-A team chronologically and cut **equal-size** windows of `W` teams with stride
`STEP` (overlapping, for a smooth trend). Each window's "time" is the date of its middle team.
Constant `W` is the whole point — it pins the data sampling error so the metrics are comparable
across time.

In [ ]:
# Exclude oversized single events that would dominate a fixed-N window. The
# Grand Champions Festival (2026-04-25) alone is 6,108 teams -- ~24% of M-A on a
# single date -- which wrecks equal-team-count windowing.
EXCLUDE_TOURNAMENT_IDS = {"69c30ae236f5b5c303dbce1c"}

tours = [t for t in tournament_ingest.load_cached_tournaments(regulation=REG)
         if t.meta.id not in EXCLUDE_TOURNAMENT_IDS]
obs = tournament_ingest.all_team_observations(tours)
obs_sorted = sorted(obs, key=lambda o: o.date[:10])
print(f"{len(obs_sorted):,} M-A teams ({len(tours)} events), "
      f"{obs_sorted[0].date[:10]} -> {obs_sorted[-1].date[:10]}")

# W = teams/window (pins statistical power); STEP = stride (resolution). Small
# STEP => many overlapping windows => smooth curve (adjacent points correlated).
# Raise STEP if the pair-model run is too slow.
W, STEP = 4000, 500
starts = list(range(0, len(obs_sorted) - W + 1, STEP))
windows = [obs_sorted[i:i + W] for i in starts]
centers = [datetime.date.fromisoformat(obs_sorted[i + W // 2].date[:10]) for i in starts]
print(f"{len(windows)} windows of {W} teams (stride {STEP}); model={MODEL}; "
      f"centers {centers[0]} -> {centers[-1]}")

## Compute the time series

One Boltzmann fit + 3-point estimate per window. Tune `W` / `FIT_KW` / `SAMPLE_KW` for
speed-vs-resolution.

In [ ]:
results = [window_metrics(win, seed=i) for i, win in enumerate(tqdm(windows, desc="windows"))]

import builtins
print(f"{'center':<12}{'N':>6}{'struct':>9}{'irred':>9}{'irr/str':>8}"
      f"{'|z|>3':>7}{'2pt-resid':>11}{'2nd-RMS':>9}")
for c, m in builtins.zip(centers, results):
    print(f"{str(c):<12}{m['n']:>6}{m['struct']:>9.5f}{m['irreducible']:>9.5f}"
          f"{m['irreducible_frac']:>8.3f}{m['frac_z3']:>7.2%}{m['fit_resid2']:>11.5f}"
          f"{m['pair2']:>9.5f}")

## Results — decomposing the 3rd order

Three panels:
1. **Total vs irreducible.** `struct` (total connected 3-point) split into the part a pairwise
   model reproduces (the shaded *gap*) and the `irreducible` part it can't. The **irreducible band
   growing in absolute terms** is the de-confounded emergence signal — the gap already absorbs
   everything the 1st/2nd-order structure predicts (via propagation), so it can't be inflated by
   the meta merely getting more structured. The faint dotted line is the 2nd-order connected RMS,
   shown only as *context* for the overall structuring (comparing raw magnitudes across orders is
   not a valid test — propagation makes 3rd a nonlinear function of 2nd).
2. **Proportional three-body-ness.** `irreducible_frac` and `|z|>3` — the *share* that's beyond
   pairwise. Scale-invariant; ~flat means irreducible grows in step with total.
3. **Fit-quality control.** Per-window 2-point reconstruction error. The irreducible rise is only
   trustworthy if this is **flat** — otherwise sharper late windows are simply harder to
   moment-match and the 2-point mismatch leaks into the 3-point residual as fake "irreducible".

In [ ]:
def series(key, res=None):
    return np.array([m[key] for m in (res if res is not None else results)])


fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1 — 3rd-order decomposition: total = irreducible + (gap = pairwise-explained).
st, ir = series("struct"), series("irreducible")
ax[0].fill_between(centers, 0, ir, alpha=0.25, color="C0", label="irreducible (beyond pairwise)")
ax[0].fill_between(centers, ir, st, alpha=0.20, color="C7", label="pairwise-explained (gap)")
ax[0].plot(centers, st, "o-", color="C3", lw=1.5, label="total (struct)")
ax[0].plot(centers, ir, "o-", color="C0", lw=1.5)
ax[0].set_ylim(bottom=0); ax[0].set_ylabel("connected 3-point magnitude")
ax[0].set_title("3rd-order: total vs irreducible")
ax[0].legend(fontsize=8, loc="upper left"); ax[0].grid(alpha=0.3)
axc = ax[0].twinx()                              # 2nd-order context (not a cross-order test)
axc.plot(centers, series("pair2"), ":", color="C2", alpha=0.6)
axc.set_ylabel("2nd-order RMS (context)", color="C2")
axc.tick_params(axis="y", labelcolor="C2"); axc.set_ylim(bottom=0)

# Panel 2 — proportional three-body-ness (scale-invariant).
ax[1].plot(centers, series("irreducible_frac"), "o-", label="irreducible fraction")
ax[1].plot(centers, series("frac_z3"), "s--", color="C1", label="|z|>3 fraction")
ax[1].set_ylim(bottom=0); ax[1].set_title("does pairwise keep up? (N fixed)")
ax[1].legend(); ax[1].grid(alpha=0.3)

# Panel 3 — fit-quality control: must stay flat for the irreducible rise to be real.
ax[2].plot(centers, series("fit_resid2"), "o-", color="C4")
ax[2].set_ylim(bottom=0)
ax[2].set_title("2-point fit residual (control)\nflat ⇒ irreducible rise is genuine")
ax[2].set_ylabel("mean |Δ⟨sᵢsⱼ⟩| over top-K"); ax[2].grid(alpha=0.3)

fig.autofmt_xdate()
fig.suptitle(f"Higher-order structure over M-A time (W={W} teams/window)")
fig.tight_layout(); plt.show()

## Null control — shuffle time

Reassign teams to windows in a random order (destroying chronology) and recompute. If the trend
above is real, this comes out **flat**; a slope here would mean the windowing itself manufactures
the signal.

In [ ]:
rng = np.random.default_rng(0)
shuffled = list(obs_sorted)
rng.shuffle(shuffled)
null_windows = [shuffled[i:i + W] for i in starts]
null_results = [window_metrics(win, seed=100 + i)
                for i, win in enumerate(tqdm(null_windows, desc="null"))]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(centers, series("struct"), "o-", color="C3", label="chronological")
ax[0].plot(centers, series("struct", null_results), "x:", color="gray", label="shuffled null")
ax[0].set_title("higher-order magnitude (struct)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(centers, series("irreducible", results), "o-", label="chronological")
ax[1].plot(centers, series("irreducible", null_results), "x:", color="gray", label="shuffled null")
ax[1].set_title("irreducible 3-body magnitude"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.autofmt_xdate()
fig.suptitle("Chronological vs shuffled-time null")
fig.tight_layout(); plt.show()

## Reading the result

Read it in this order:

1. **Fit-control first (panel 3).** If the 2-point residual drifts upward with time, stop — the
   "irreducible" rise is partly fit error on the sharper late windows, not structure. Only if it's
   flat do the next two panels mean what they say.
2. **Emergence = the irreducible band (panel 1).** The shaded gap is everything the growing
   1st/2nd-order structure already explains by propagation; the `irreducible` band is what's left.
   Its growing in absolute terms — over the flat shuffled null below — is the de-confounded claim
   that genuinely beyond-pairwise structure emerged, *not* merely that the meta got more
   structured. (Raw `struct` rising alone would be confounded; this isn't.)
3. **Proportion (panel 2).** A flat `irreducible_frac` means irreducible and total grew at the
   same rate — the meta's *proportional* three-body-ness is scale-invariant. That's a distinct,
   also-interesting statement from "did it emerge"; don't conflate the two.

So the honest one-liner is: *if the fit-control is flat, the absolute amount of beyond-pairwise
3-body structure grew as M-A matured, while its share of the (also-growing) total held roughly
constant.*

**Caveat on levels.** The `se` (hence `struct`, `irreducible`, `|z|`) assumes iid teams and iid
MCMC draws; real teams cluster by event and MCMC autocorrelates, so magnitudes are mildly
over-stated. The **trend**, the **null**, and the **fit-control** are robust to this; absolute
numbers are not. Harden with bigger `SAMPLE_KW`, larger `W`, and/or a block-bootstrap over events.

Natural extensions: repeat on the species+item model (item-resolved cores; mind support-gating),
and re-run on **M-B** as it accrues data to watch a second meta crystallize from `t = 0`.